# Notebook 05 — Modelo LSTM para predicción de huella de carbono

En este notebook implementamos una red neuronal LSTM para predecir
la huella de carbono operacional del consumo eléctrico español
con resolución de 15 minutos y horizontes de 48h y 72h.

**Enfoque**: predicción directa multihorizonte — la red predice
todos los pasos del horizonte de una sola vez, sin acumulación
de errores.

**Entrada**: ventana de 2 días (192 pasos) de la serie histórica.  
**Salida**: predicción de 48h (192 pasos) o 72h (288 pasos).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Reproducibilidad total
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)

Device: cpu
PyTorch: 2.10.0+cpu


## 1. Carga de datos

Se cargan los mismos parquets generados en el notebook 01,
manteniendo la misma división temporal que el resto de modelos:
- Train: 2022–2023
- Validación: 2024
- Test: 2025

In [2]:
DATA_DIR = Path.cwd() / "data_processed"

train = pd.read_parquet(DATA_DIR / "train_2022_2023.parquet")
val   = pd.read_parquet(DATA_DIR / "val_2024.parquet")
test  = pd.read_parquet(DATA_DIR / "test_2025.parquet")

y_train = train["y"].astype(float).values
y_val   = val["y"].astype(float).values
y_test  = test["y"].astype(float).values

print("Train:", y_train.shape)
print("Val:  ", y_val.shape)
print("Test: ", y_test.shape)

Train: (70080,)
Val:   (35136,)
Test:  (35040,)


## 2. Normalización

Normalizamos la serie al rango [0,1]
con MinMaxScaler ajustado **únicamente sobre el train**.

Esto es crítico: si ajustásemos sobre todos los datos estaríamos
filtrando información futura al modelo (data leakage).

La desnormalización se aplica después de predecir para obtener
las métricas en unidades originales (gCO2eq/kWh).

In [3]:
# Scaler ajustado solo sobre train — nunca sobre val ni test
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(y_train.reshape(-1, 1))

y_train_sc = scaler.transform(y_train.reshape(-1, 1)).flatten()
y_val_sc   = scaler.transform(y_val.reshape(-1, 1)).flatten()
y_test_sc  = scaler.transform(y_test.reshape(-1, 1)).flatten()

print("Train escalado — min:", y_train_sc.min().round(4),
      "max:", y_train_sc.max().round(4))
print("Val escalado   — min:", y_val_sc.min().round(4),
      "max:", y_val_sc.max().round(4))

Train escalado — min: 0.0 max: 1.0
Val escalado   — min: 0.0004 max: 0.835


## 3. Configuración de horizontes y ventana de entrada

Se definen los parámetros temporales del problema.

La ventana de entrada (lookback) se fija en 2 días (192 pasos),
justificado por los picos de autocorrelación observados en el EDA
cada 96 pasos (1 día). Dos días capturan el patrón diario completo
con suficiente contexto histórico reciente.

In [4]:
FREQ_MIN        = 15
STEPS_PER_HOUR  = 60 // FREQ_MIN   # 4
SEASONAL_PERIOD = 24 * STEPS_PER_HOUR  # 96 pasos = 1 día

# Ventana de entrada: 2 días
LOOKBACK = 2 * SEASONAL_PERIOD  # 192 pasos

# Horizontes de predicción
HORIZONS = {
    "48h": 48 * STEPS_PER_HOUR,  # 192 pasos
    "72h": 72 * STEPS_PER_HOUR,  # 288 pasos
}

print(f"Lookback:      {LOOKBACK} pasos = {LOOKBACK * FREQ_MIN / 60:.0f} horas")
print(f"Horizonte 48h: {HORIZONS['48h']} pasos")
print(f"Horizonte 72h: {HORIZONS['72h']} pasos")

Lookback:      192 pasos = 48 horas
Horizonte 48h: 192 pasos
Horizonte 72h: 288 pasos


## 4. Dataset con ventana deslizante

Transformamos la serie temporal
en un problema supervisado mediante ventana deslizante (sliding window).

Cada muestra tiene:
- X: 192 pasos históricos  ->  shape (192, 1)
- y: H pasos futuros       ->  shape (H,)

El Dataset lko creamos por separado para cada horizonte ya que
el tamaño de la salida es diferente (192 vs 288 pasos).

In [ ]:
class SlidingWindowDataset(Dataset):
    """
    Transforma una serie temporal en pares (ventana_entrada, horizonte_objetivo)
    mediante ventana deslizante — Brownlee (2019), cap. 4.
    
    Args:
        series:   array (N,) normalizado
        lookback: pasos de contexto de entrada
        horizon:  pasos a predecir
    """
    def __init__(self, series, lookback, horizon):
        self.series   = series
        self.lookback = lookback
        self.horizon  = horizon
        # Número de muestras posibles
        self.n = len(series) - lookback - horizon + 1

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        x = self.series[idx : idx + self.lookback]          # (lookback,)
        y = self.series[idx + self.lookback :
                        idx + self.lookback + self.horizon] # (horizon,)
        # LSTM espera un  shape (lookback, features) -> añadimos dimensión
        x = torch.tensor(x, dtype=torch.float32).unsqueeze(-1)  # (192, 1)
        y = torch.tensor(y, dtype=torch.float32)                 # (H,)
        return x, y

# Verificación de shapes
ds = SlidingWindowDataset(y_train_sc, LOOKBACK, HORIZONS["48h"])
x0, y0 = ds[0]
print(f"Muestras en train: {len(ds)}")
print(f"X shape: {x0.shape}   ->  (lookback, features)")
print(f"y shape: {y0.shape}   ->  (horizon,)")

Muestras en train: 69697
X shape: torch.Size([192, 1])   ->  (lookback, features)
y shape: torch.Size([192])   ->  (horizon,)
